In [3]:
# Importing modules
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS

In [ ]:
water_path = PATHS['cleaned_data_notebooks']/ 'water_cleaned.parquet'
water_non_merged_pd = pd.read_parquet(water_path)
water_non_merged_pd.head()

,date,storage,id,storage_imputed
0,1988-01-05,103,1,0
1,1988-01-12,103,1,0
2,1988-01-19,103,1,0
3,1988-01-26,103,1,0
4,1988-02-02,103,1,0


In [10]:
water_pd.isna().sum()

date               0
storage            0
id                 0
storage_imputed    0
dtype: int64

In [11]:
reservoirs_path = PATHS['cleaned_data_notebooks']/ 'reservoirs_merged.csv'
reservoirs_pd = pd.read_csv(reservoirs_path)
reservoirs_pd.head()

,id,scope,name,capacity,electric_flag,latitude,longitude,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
0,1,guadalquivir,brena,103,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cordoba,andalucia,NaN,NaN,NaN,NaN
1,3,guadalquivir,fernandina,247,0,38.179646,-3.570224,guadalquivir,rio guarrizas,NaN,NaN,NaN,jaen,andalucia,presa fabrica gravedad (hormigon vibrado),719.55,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,5,guadalquivir,puebla cazalla,87,0,37.129772,-5.243309,guadalquivir,rio corbones,NaN,NaN,NaN,sevilla,andalucia,presa fabrica gravedad (hormigon compactado),218.25,NaN,https://sig.mapama.gob.es/WebServices/clientew...
3,6,guadalquivir,pedro marin,19,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,jaen,andalucia,NaN,NaN,NaN,NaN
4,9,cuenca mediterranea andaluza,vinuela,170,0,36.860400,-4.164435,cuencas mediterraneas andaluzas,rio guaro,https://www.google.com/search?kgmid=/g/120jr2sh,NaN,https://www.wikidata.org/wiki/Q5830515,malaga,andalucia,presa materiales sueltos pantalla hormigon,426.00,NaN,https://sig.mapama.gob.es/WebServices/clientew...


In [12]:
reservoirs_pd.isna().sum()

id                        0
scope                     0
name                      0
capacity                  0
electric_flag             0
latitude                104
longitude               104
basin                   104
riverbed                104
google                  303
openstreetmap           372
wikidata                243
province                  0
autonomous_community      0
type                    104
crest_elevation         104
dam_height              256
report                  104
dtype: int64

Merging the dataframes

In [13]:
water_pd = pd.merge(water_pd, reservoirs_pd[['id', 'capacity', 'crest_elevation', 'province', 'autonomous_community']], on='id', how='left')
water_pd.head()

,date,storage,id,storage_imputed,capacity,crest_elevation,province,autonomous_community
0,1988-01-05,103,1,0,103,NaN,cordoba,andalucia
1,1988-01-12,103,1,0,103,NaN,cordoba,andalucia
2,1988-01-19,103,1,0,103,NaN,cordoba,andalucia
3,1988-01-26,103,1,0,103,NaN,cordoba,andalucia
4,1988-02-02,103,1,0,103,NaN,cordoba,andalucia


### Adding year, month and day features

In [14]:
water_pd_full['year'] = pd.to_datetime(water_pd_full['date']).dt.year
water_pd_full['month'] = pd.to_datetime(water_pd_full['date']).dt.month
water_pd_full['day'] = pd.to_datetime(water_pd_full['date']).dt.day
water_pd_full.head()

,date,storage,capacity,crest_elevation,province,autonomous_community,id,year,month,day
0,1988-01-05,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,5
1,1988-01-12,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,12
2,1988-01-19,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,19
3,1988-01-26,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,26
4,1988-02-02,103.0,103.0,NaN,cordoba,andalucia,1,1988,2,2


### Adding lag features

In [16]:
for lag in [1,2,3,4]:
    water_pd_full[f'storage_last_week_{lag}'] = water_pd_full.groupby('id')['storage'].shift(lag)
water_pd_full['storage_last_year'] = water_pd_full.groupby('id')['storage'].shift(52)
water_pd_full.head(15)

,date,storage,capacity,crest_elevation,province,autonomous_community,id,year,month,day,storage_missing,storage_last_week_1,storage_last_week_2,storage_last_week_3,storage_last_week_4,storage_last_year
0,1988-01-05,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,5,0,NaN,NaN,NaN,NaN,NaN
1,1988-01-12,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,12,0,103.0,NaN,NaN,NaN,NaN
2,1988-01-19,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,19,0,103.0,103.0,NaN,NaN,NaN
3,1988-01-26,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,26,0,103.0,103.0,103.0,NaN,NaN
4,1988-02-02,103.0,103.0,NaN,cordoba,andalucia,1,1988,2,2,0,103.0,103.0,103.0,103.0,NaN
5,1988-02-09,103.0,103.0,NaN,cordoba,andalucia,1,1988,2,9,0,103.0,103.0,103.0,103.0,NaN
6,1988-02-16,103.0,103.0,NaN,cordoba,andalucia,1,1988,2,16,0,103.0,103.0,103.0,103.0,NaN
7,1988-02-23,103.0,103.0,NaN,cordoba,andalucia,1,1988,2,23,0,103.0,103.0,103.0,103.0,NaN
8,1988-03-01,103.0,103.0,NaN,cordoba,andalucia,1,1988,3,1,0,103.0,103.0,103.0,103.0,NaN
9,1988-03-08,103.0,103.0,NaN,cordoba,andalucia,1,1988,3,8,0,103.0,103.0,103.0,103.0,NaN


In [17]:
water_pd_full.isna().sum()

date                         0
storage                      0
capacity                   678
crest_elevation         147626
province                   678
autonomous_community       678
id                           0
year                         0
month                        0
day                          0
storage_missing              0
storage_last_week_1        401
storage_last_week_2        802
storage_last_week_3       1203
storage_last_week_4       1604
storage_last_year        20852
dtype: int64

### Rolling average and standard deviation for last month

In [18]:
water_pd_full['storage_mean_4w'] = water_pd_full.groupby('id')['storage'].rolling(4, min_periods=1).mean().reset_index(level=0, drop=True)
water_pd_full['storage_std_4w'] = water_pd_full.groupby('id')['storage'].rolling(4, min_periods=1).std().reset_index(level=0, drop=True)

In [19]:
water_pd_full.head(15)

,date,storage,capacity,crest_elevation,province,autonomous_community,id,year,month,day,storage_missing,storage_last_week_1,storage_last_week_2,storage_last_week_3,storage_last_week_4,storage_last_year,storage_mean_4w,storage_std_4w
0,1988-01-05,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,5,0,NaN,NaN,NaN,NaN,NaN,103.00,NaN
1,1988-01-12,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,12,0,103.0,NaN,NaN,NaN,NaN,103.00,0.000000
2,1988-01-19,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,19,0,103.0,103.0,NaN,NaN,NaN,103.00,0.000000
3,1988-01-26,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,26,0,103.0,103.0,103.0,NaN,NaN,103.00,0.000000
4,1988-02-02,103.0,103.0,NaN,cordoba,andalucia,1,1988,2,2,0,103.0,103.0,103.0,103.0,NaN,103.00,0.000000
5,1988-02-09,103.0,103.0,NaN,cordoba,andalucia,1,1988,2,9,0,103.0,103.0,103.0,103.0,NaN,103.00,0.000000
6,1988-02-16,103.0,103.0,NaN,cordoba,andalucia,1,1988,2,16,0,103.0,103.0,103.0,103.0,NaN,103.00,0.000000
7,1988-02-23,103.0,103.0,NaN,cordoba,andalucia,1,1988,2,23,0,103.0,103.0,103.0,103.0,NaN,103.00,0.000000
8,1988-03-01,103.0,103.0,NaN,cordoba,andalucia,1,1988,3,1,0,103.0,103.0,103.0,103.0,NaN,103.00,0.000000
9,1988-03-08,103.0,103.0,NaN,cordoba,andalucia,1,1988,3,8,0,103.0,103.0,103.0,103.0,NaN,103.00,0.000000


In [12]:
water_engineered_path = PATHS['engineered_data'] / 'water_engineered.parquet'
water_pd_full = pd.read_parquet(water_engineered_path)
water_pd_full.head()

,date,storage,capacity,crest_elevation,province,autonomous_community,id,year,month,day,storage_missing,storage_last_week_1,storage_last_week_2,storage_last_week_3,storage_last_week_4,storage_last_year,storage_mean_4w,storage_std_4w,completeness,week_idx
0,1988-01-05,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,5,0,NaN,NaN,NaN,NaN,NaN,103.0,NaN,1.0,1
1,1988-01-12,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,12,0,103.0,NaN,NaN,NaN,NaN,103.0,0.0,1.0,2
2,1988-01-19,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,19,0,103.0,103.0,NaN,NaN,NaN,103.0,0.0,1.0,3
3,1988-01-26,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,26,0,103.0,103.0,103.0,NaN,NaN,103.0,0.0,1.0,4
4,1988-02-02,103.0,103.0,NaN,cordoba,andalucia,1,1988,2,2,0,103.0,103.0,103.0,103.0,NaN,103.0,0.0,1.0,5


In [14]:
engineered_water_path = PATHS['engineered_data'] / 'water_engineered.parquet'
engineered_water_path.parent.mkdir(parents=True, exist_ok=True)
water_pd_full.to_parquet(engineered_water_path, index=False)